# EM-11 model comparison, notebook 1 of 2: prepare the clips

Finds the FSD50K clips the comparison uses (`docs/em-11-compare-clips.json`) in the Kaggle copy,
checks the copy's audio against clips whose audio is known, converts every clip to 16 kHz and packs
them into `em11-data.tar` for notebook 2. Nothing here needs a GPU.

**Settings (right-hand panel)**

- Accelerator: **None** (CPU).
- Internet: **On** (code from GitHub, labels from Zenodo, a few clips from the Hugging Face mirror).
- Input: **Add Input → Datasets → `yousirui1/fsd50k`**, version 1.
- Environment: *Pin to original environment*.

Run it with **Save Version → Save & Run All (Commit)**: the output is kept as a version and becomes
notebook 2's input. First with `SMOKE = True` (3 clips per group, a few minutes), then with
`SMOKE = False` as a new version (about 30-60 minutes). `docs/em-11-model-comparison.md` explains every step.

In [ ]:
REPO = "https://github.com/SWE-encore/encoremoment.git"
REF = "feat/em-11-model-comparison"  # the branch, tag or commit to run
SMOKE = True                          # True: 3 clips per group, to try the whole path; then False
POLICY = "auto"                       # auto: use the copy only if its audio matches; mirror: never use it

In [ ]:
import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from IPython.display import Markdown, display


def sh(*cmd, check=True, cwd=None):
    """Run one command, its output shown here (a child's own output would not reach the notebook)."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() and check:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode


def out(*cmd, cwd=None):
    return subprocess.check_output([str(c) for c in cmd], text=True, cwd=cwd).strip()


def find(name, root=Path("/kaggle/input"), depth=4, dirs=False):
    """Paths under /kaggle/input whose name contains ``name``, nearest first."""
    hits = [p for d in range(1, depth + 1) for p in root.glob("/".join("*" * d))
            if name in p.name.lower() and (p.is_dir() or not dirs)]
    return sorted(hits, key=lambda p: len(p.parts))


UV = [sys.executable, "-m", "uv"]


def venv(name, requirements):
    """A clean Python 3.12 environment with pinned packages; what it holds is saved next to the results."""
    path = Path(f"/tmp/venv-{name}")
    shutil.rmtree(path, ignore_errors=True)
    sh(*UV, "venv", "--quiet", "--python", "3.12", path)
    sh(*UV, "pip", "install", "--quiet", "--python", path / "bin/python", "-r", CODE / requirements)
    freeze = out(*UV, "pip", "freeze", "--python", path / "bin/python")
    (WORK / f"env-{name}.txt").write_text(freeze + "\n")
    return path / "bin/python"

WORK, CODE, DATA = Path("/kaggle/working"), Path("/tmp/encoremoment"), Path("/tmp/em11-data")

## Code and environment

In [ ]:
shutil.rmtree(CODE, ignore_errors=True)
sh("git", "clone", "--quiet", REPO, CODE)
sh("git", "-C", CODE, "checkout", "--quiet", REF)
COMMIT = out("git", "-C", CODE, "rev-parse", "HEAD")
print("code at", COMMIT)

sh(sys.executable, "-m", "pip", "install", "--quiet", "uv==0.9.5")
PY = venv("base", "server/tools/kaggle/requirements-base.txt")
print(out("ffmpeg", "-version").splitlines()[0])
print(os.cpu_count(), "CPUs")

## The Kaggle copy of FSD50K

In [ ]:
copies = find("fsd50k", dirs=True)
if not copies and POLICY != "mirror":
    raise SystemExit("Add Input: the dataset yousirui1/fsd50k (version 1)")
SOURCE = copies[0] if copies else None
print("copy:", SOURCE)
if SOURCE:
    print(sorted(p.name for p in SOURCE.iterdir()))

## Prepare

Checks the copy first (known clips, then a sample against the pinned mirror), then converts every
chosen clip. A clip the copy lacks, or every clip if the copy fails a check, comes from the mirror.

In [ ]:
shutil.rmtree(DATA, ignore_errors=True)
args = ["prepare", "--out", DATA, "--policy", POLICY, "--workers", 2 * os.cpu_count()]
if SOURCE:
    args += ["--source", SOURCE]
if SMOKE:
    args += ["--limit", 3]
started = time.time()
sh(PY, CODE / "server/tools/em11_compare.py", *args)
print(f"{(time.time() - started) / 60:.0f} minutes")

## Pack for notebook 2

In [ ]:
(DATA / "code.json").write_text(json.dumps({"repo": REPO, "ref": REF, "commit": COMMIT, "smoke": SMOKE}, indent=1) + "\n")
with tarfile.open(WORK / "em11-data.tar", "w") as tar:  # one file: notebook outputs hold few files
    tar.add(DATA, arcname="em11-data")
for name in ("manifest.json", "verify-report.md", "code.json"):
    shutil.copy(DATA / name, WORK / name)
print(f"em11-data.tar: {(WORK / 'em11-data.tar').stat().st_size / 1e9:.2f} GB")
display(Markdown((DATA / "verify-report.md").read_text()))